In [1]:
import pandas as pd
import numpy as np
import re
import nltk
import string
import nltk
from nltk.corpus import stopwords
import statistics
import tensorflow
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,LSTM,Dense,Dropout,SimpleRNN,GRU,Bidirectional
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix

In [2]:
import os
print(os.getcwd())
print(os.listdir())

c:\Users\SwapnilShinde\OneDrive\Documents\Study Material\Sem 6\AML\Experiment Codes\Sentiment-Analysis-using-Recurrent-Neural-Networks-main
['.gitattributes', '.github', 'data', 'IMDB Dataset.csv', 'LICENSE', 'main.ipynb', 'ml_lab_env', 'README.md', 'requirements.txt']


In [3]:
df = pd.read_csv('IMDB Dataset.csv',nrows=25000)

In [4]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [5]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\SwapnilShinde\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


True

In [6]:
def remove_html_tags(text):
  clean_text = re.sub(r'<.*?>', '', text)
  return clean_text

df['review'] = df['review'].apply(remove_html_tags)
df.head()


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. The filming tec...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [7]:
import string
punc = string.punctuation
punc

'!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'

In [8]:
def remove_punc(text):
  clean_text = "".join([char for char in text if char not in punc])
  return clean_text

df['review'] = df['review'].apply(remove_punc)
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production The filming tech...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically theres a family where a little boy J...,negative
4,Petter Matteis Love in the Time of Money is a ...,positive


In [9]:
df['review'] =  df['review'].str.lower()

In [11]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

def stp_words(text):
    return " ".join([word for word in text.split() if word not in stop_words])

df['review'] = df['review'].apply(stp_words)

In [12]:
token = Tokenizer()
token.fit_on_texts(df['review'])

In [13]:
len(token.word_index)

144060

In [14]:
seq = token.texts_to_sequences(df['review'])

In [15]:
average = statistics.mean

In [16]:
average([len(x) for x in seq])

119.99108

In [17]:
padding = pad_sequences(seq,maxlen=220,padding='pre')

In [18]:
padding.shape

(25000, 220)

In [19]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['sentiment'] = le.fit_transform(df['sentiment'])

In [20]:
x= padding
y = df['sentiment']

In [21]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.175, random_state=42)

In [22]:
x_train.shape , x_test.shape , y_train.shape , y_test.shape

((20625, 220), (4375, 220), (20625,), (4375,))

In [23]:
y.value_counts()

sentiment
0    12526
1    12474
Name: count, dtype: int64

In [24]:
vocab_size = len(token.word_index) + 1
print(f"Vocabulary size in x_train: {vocab_size}")

Vocabulary size in x_train: 144061


# **Model** **Building** ⌛

In [26]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

m1 = Sequential()
m1.add(Embedding(67142, 100))   # removed input_length
m1.add(SimpleRNN(150, return_sequences=True))
m1.add(SimpleRNN(50, return_sequences=True))
m1.add(SimpleRNN(25))
m1.add(Dense(1, activation='sigmoid'))

m1.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_3 (SimpleRNN)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_4 (SimpleRNN)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_5 (SimpleRNN)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [29]:
vocab_size = len(token.word_index) + 1

In [30]:
Embedding(vocab_size, 100)

<Embedding name=embedding_2, built=False>

In [31]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

m1 = Sequential()
m1.add(Embedding(vocab_size, 100))   # ✅ fixed
m1.add(SimpleRNN(128, return_sequences=True))
m1.add(SimpleRNN(64))
m1.add(Dense(1, activation='sigmoid'))

m1.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_6 (SimpleRNN)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_7 (SimpleRNN)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
m2 = Sequential()
m2.add(Embedding(144771,output_dim=250,input_length=220))
m2.add(Bidirectional(LSTM(150,return_sequences=True)))
m2.add(Bidirectional(GRU(100,return_sequences=True)))
m2.add(Dropout(0.2))
m2.add(Bidirectional(LSTM(50,return_sequences=True)))
m2.add(Bidirectional(GRU(30,return_sequences=True)))
m2.add(Dropout(0.2))
m2.add(LSTM(10))
m2.add(Dense(1,activation='sigmoid'))

m2.summary()

Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_2 (Embedding)     (None, 220, 250)          36192750  
                                                                 
 bidirectional (Bidirection  (None, 220, 300)          481200    
 al)                                                             
                                                                 
 bidirectional_1 (Bidirecti  (None, 220, 200)          241200    
 onal)                                                           
                                                                 
 dropout (Dropout)           (None, 220, 200)          0         
                                                                 
 bidirectional_2 (Bidirecti  (None, 220, 100)          100400    
 onal)                                                           
                                                      

In [32]:
m1.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [33]:
sequences = token.texts_to_sequences(df['review'])

In [34]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

X = pad_sequences(sequences, maxlen=220)

In [35]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    X, df['sentiment'], test_size=0.2, random_state=42
)

In [36]:
history = m1.fit(
    x_train, y_train,
    epochs=5,
    batch_size=64,
    validation_data=(x_test, y_test)
)

Epoch 1/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 53s 159ms/step - accuracy: 0.6029 - loss: 0.6404 - val_accuracy: 0.5346 - val_loss: 0.6762
Epoch 2/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 68s 218ms/step - accuracy: 0.7447 - loss: 0.5142 - val_accuracy: 0.7072 - val_loss: 0.5749
Epoch 3/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 49s 156ms/step - accuracy: 0.8556 - loss: 0.3428 - val_accuracy: 0.7920 - val_loss: 0.4826
Epoch 4/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 49s 155ms/step - accuracy: 0.8895 - loss: 0.2738 - val_accuracy: 0.7904 - val_loss: 0.5386
Epoch 5/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 65s 207ms/step - accuracy: 0.9367 - loss: 0.1687 - val_accuracy: 0.7866 - val_loss: 0.6513


In [37]:
loss, acc = m1.evaluate(x_test, y_test)
print("Accuracy:", acc)

157/157 ━━━━━━━━━━━━━━━━━━━━ 3s 22ms/step - accuracy: 0.7866 - loss: 0.6513
Accuracy: 0.7865999937057495


In [38]:
pred = m1.predict(x_test)

# Convert probabilities → 0/1
pred = (pred > 0.5).astype(int)

157/157 ━━━━━━━━━━━━━━━━━━━━ 5s 33ms/step


In [39]:
from sklearn.metrics import confusion_matrix

print(confusion_matrix(y_test, pred))

[[1945  589]
 [ 478 1988]]


In [40]:
m3 = Sequential()
m3.add(Embedding(144771,output_dim=250,input_length=220))
m3.add(Bidirectional(LSTM(170,return_sequences=True)))
m3.add(Bidirectional(GRU(130,return_sequences=True)))
m3.add(Dropout(0.2))
m3.add(Bidirectional(LSTM(70,return_sequences=True)))
m3.add(Bidirectional(GRU(40,return_sequences=True)))
m3.add(Dropout(0.2))
m3.add(Bidirectional(GRU(20,return_sequences=True)))
m3.add(LSTM(20))
m3.add(Dense(1,activation='sigmoid'))

m3.summary()

c:\Users\SwapnilShinde\OneDrive\Documents\Study Material\Sem 6\AML\Experiment Codes\Sentiment-Analysis-using-Recurrent-Neural-Networks-main\ml_lab_env\lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_3 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_4 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

**m3 model delivered an accuracy of 99.90 after 15 epochs.**

Others metrics after 15 epochs are:


*   loss : 0.38
*   validation accuracy : 84.73
*   validation loss : 86.81



# **Evaluation** ⚓

In [ ]:
text = ['''One of the most brain dead idiotic movies with the most crazy and psychotic characters as protagonists of 2024.
What a waste of time and energy. There are 0 characters in this movie that are remotely normal''' ]
seq = token.texts_to_sequences(text)
padding = pad_sequences(seq,maxlen=220,padding='pre')
pred = mode.predict(padding)
if pred < 0.2:
    print('flop movie')
elif 0.2 <= pred < 0.55:
    print('average movie')
elif 0.55 <= pred < 0.8:
    print('good movie')
else:
    print('Blockboster movie')

1/1 [==============================] - 0s 48ms/step
flop movie


In [ ]:
text = ['''Laapata Ladies is a sort of film that is very rare these days. A satire that is an eye-opener for the audience and needs everyone's attention. With unknown cast, limited budget and great script, it's a movie for all age-groups, every gender and is bound to go a long way.

This Kiran Rao's tragicomedy of two brides in rural India who accidentally get swapped during a train journey while returning from their wedding. Their misadventures after this strange event will make you laugh and the same time think about the social taboos a women has to go through in our society.

Ravi Kishan playing the cop investigating this case is a delight to watch. Very subtly and hilariously he gives movie the much needed comic side that perfectly balances the emotional cause for which the movie was made. Probably one of his best performance till date.
''']

In [ ]:
seq = token.texts_to_sequences(text)
padding = pad_sequences(seq,maxlen=220,padding='pre')
pred = mode.predict(padding)
if pred < 0.2:
    print('flop movie')
elif 0.2 <= pred < 0.55:
    print('average movie')
elif 0.55 <= pred < 0.8:
    print('good movie')
else:
    print('Blockboster movie')

1/1 [==============================] - 0s 69ms/step
Blockboster movie
